# Wine Market Project — Data Pipeline

This notebook cleans the raw wine data and writes
processed files for a **Tableau dashboard**, a **Plotly HTML data story**,
and an **Inkscape SVG poster**.

**Sections**
1. Setup
2. Clean OIV production/consumption
3. Clean WineMag reviews
4. Clean varietal region area (2000/2010/2016/2023)
5. Parse country yield tables (.txt)
6. Estimate variety production
7. Wine style / color for reviews
8. Tableau files
9. Poster files
10. Validation report


## Setup

In [2]:
# Standard library + pandas/numpy only. Plotly will be imported later.

import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

# Setting up file organization so that later on everything goes to the right place!
ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TABLEAU = ROOT / "tableau"
HTML = ROOT / "html"
SVG = ROOT / "svg"
DOCS = ROOT / "docs"

# checking that i have everything all set up correctly
for d in (RAW, PROC, TABLEAU, HTML, SVG, DOCS):
    d.mkdir(parents=True, exist_ok=True)

# Project palette based off of a wine/grape aestetic to stay on theme
# with typewriter font, soft eggshell background, muted wine tones for sophisication (also on theme)
PALETTE = {
    "background": "#F2E9DC",   # soft beige / eggshell
    "panel":      "#FBF6EC",   # lighter cream for cards/panels
    "ink":        "#3A2E2A",   # dark brown text
    "wine_dark":  "#5E2129",   # deep muted wine
    "wine":       "#7B3B47",   # muted wine (primary)
    "wine_light": "#A86A74",   # dusty rose
    "gold":       "#B79268",   # muted gold / neutral accent
    "grid":       "#D8CBB5",   # faint gridlines on eggshell
}
FONT_FAMILY = "Courier New, Courier, monospace"   # typewriter style everywhere

# Wine-color swatches reused by charts that split red / white / etc.
COLOR_BY_STYLE = {
    "red":          PALETTE["wine_dark"],
    "white":        PALETTE["gold"],
    "rose":         PALETTE["wine_light"],
    "sparkling":    "#C9A24B",
    "other":        "#9C8F7A",
    "unknown":      "#BCB2A1",
    "grey_or_gris": "#8C8C8C",
}


print("Setup OK")
print("Project root:", ROOT)
print("Raw files present:", sorted(p.name for p in RAW.glob("*")))


Setup OK
Project root: C:\Users\jtrob\OneDrive\Desktop\wine-market-project
Raw files present: ['country_yield_tables.txt', 'oiv_wine.xlsx', 'old_new_world_mapping.xlsx', 'variety_area_by_country.xlsx', 'variety_region_area_2000.xlsx', 'variety_region_area_2010.xlsx', 'variety_region_area_2016.xlsx', 'variety_region_area_2023.xlsx', 'winemag-data-130k-v2.csv']


In [3]:
# helper formula #1
# to use as we parse through the data and want to get a quick easy to see dimensions of the dataframe
def check(label, df):
    print(f"{label}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"columns: {list(df.columns)}")

# helper formula #2
# to rename columns super easy and quick so everything is standardized
def to_snake(col):
    """lower-case, replace spaces/slashes/non-breaking-spaces, strip junk."""
    c = str(col).strip().lower().replace("/", "_").replace(" ", "_").replace("\xa0", "_")
    c = re.sub(r"[^0-9a-z_]", "", c)
    return re.sub(r"_+", "_", c).strip("_")

# Cleaning

### Clean OIV production & consumption

Read the OIV export, tidy columns, and build the
quantity fields. Save both a **long** table (one row per country/year/variable)
and a **wide** table (production and consumption as columns).

In [4]:
oiv = pd.read_excel(RAW / "oiv_wine.xlsx")

oiv.columns = [to_snake(c) for c in oiv.columns]

oiv

C:\Users\jtrob\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,continent,region_country,product,variable,year,unit,quantity,flag
0,Asia,Afghanistan,Wine,Consumption,1995.0,1000 hl,0.0,NaN
1,Asia,Afghanistan,Wine,Production,1995.0,1000 hl,0.0,NaN
2,Asia,Afghanistan,Wine,Consumption,1996.0,1000 hl,0.0,NaN
3,Asia,Afghanistan,Wine,Production,1996.0,1000 hl,0.0,NaN
4,Asia,Afghanistan,Wine,Consumption,1997.0,1000 hl,0.0,NaN
...,...,...,...,...,...,...,...,...
12124,Africa,Zimbabwe,Wine,Consumption,2024.0,1000 hl,23.0,NaN
12125,Africa,Zimbabwe,Wine,Production,2024.0,1000 hl,14.0,NaN
12126,Africa,Zimbabwe,Wine,Production,2025.0,1000 hl,14.0,NaN
12127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
oiv = oiv.rename(columns={"region_country": "country"})

# removing blank footer rows
n_before = len(oiv)
oiv = oiv[oiv["product"] == "Wine"].copy()
print(f"kept Wine rows: {len(oiv):,} (dropped {n_before - len(oiv)} non-Wine/blank rows)")

# standardizing fields
oiv["country"]  = oiv["country"].astype(str).str.strip()
oiv["variable"] = oiv["variable"].astype(str).str.strip().str.title()   # Production / Consumption
oiv["year"]     = pd.to_numeric(oiv["year"], errors="coerce").astype("Int64")

# OIV reports quantity in 1000 hl; multiply by 1000 to store plain hectolitres (hl)
oiv["quantity_hl"] = oiv["quantity"] * 1000

# wide table: one row per country-year, production & consumption side by side.
# (drops the units/product/flag columns -- redundant or all-NaN.)
oiv_cleaned = (oiv.pivot_table(index=["country", "year"], columns="variable",
                               values="quantity_hl", aggfunc="sum").reset_index())
oiv_cleaned.columns.name = None
oiv_cleaned = oiv_cleaned.rename(columns={"Consumption": "consumption_hl",
                                          "Production":  "production_hl"})

check("oiv_cleaned", oiv_cleaned)
oiv_cleaned

kept Wine rows: 12,127 (dropped 2 non-Wine/blank rows)
oiv_cleaned: 6,591 rows x 4 cols
columns: ['country', 'year', 'consumption_hl', 'production_hl']


,country,year,consumption_hl,production_hl
0,Afghanistan,1995,0.0,0.0
1,Afghanistan,1996,0.0,0.0
2,Afghanistan,1997,0.0,0.0
3,Afghanistan,1998,0.0,0.0
4,Afghanistan,1999,0.0,0.0
...,...,...,...,...
6586,Zimbabwe,2021,22000.0,14000.0
6587,Zimbabwe,2022,22000.0,14000.0
6588,Zimbabwe,2023,26000.0,15000.0
6589,Zimbabwe,2024,23000.0,14000.0


visual check looks good as i referenced manually zimbabwe 2024 consumption and production from original and cleaned so lets save it!

In [6]:
oiv_cleaned.to_csv(PROC / "oiv_cleaned.csv", index=False)

print("saved: oiv_cleaned.csv")

saved: oiv_cleaned.csv


### Clean WineMag reviews

Drop the index column, keep the useful columns, and compute `value_score = points / price`

In [7]:
reviews = pd.read_csv(RAW / "winemag-data-130k-v2.csv")
reviews.columns = [to_snake(c) for c in reviews.columns]

reviews.head(4)

,unnamed_0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian


In [8]:
# tidying up
reviews = reviews.drop(columns=[c for c in reviews.columns if c.lower().startswith("unnamed")])

# parsing out the wine's vintage year by finding the LAST plausible 4-digit year (1600-2026) found in the title
# for example: 1912 Winemakers 2009 Piorro Grande Reserva will return 2009 not 1912 -- i.e., the later of the two
def extract_vintage(title):
    if not isinstance(title, str):
        return np.nan
    matches = re.findall(r"\b(16\d{2}|20[0-1]\d|202[0-6])\b", title)
    return int(matches[-1]) if matches else np.nan

reviews["vintage_year"] = reviews["title"].apply(extract_vintage).astype("Int64")

# creating a value_score: quality per dollar (edge case: only when we actually have a positive price --> i.e. don't calculate for 0 or null values)
reviews["value_score"] = np.where(reviews["price"] > 0,
                                  reviews["points"] / reviews["price"], np.nan)


# tidying up & organizing
reviews["region"] = reviews["region_1"]
reviews["sub_region"] = reviews["region_2"]

keep_cols = ["points", "title", "winery", "variety", "designation", "country", "province", "region", "sub_region",
              "price", "value_score", "vintage_year", "description"]

reviews = reviews[keep_cols].copy()

reviews["points"] = pd.to_numeric(reviews["points"], errors="coerce")
reviews["price"]  = pd.to_numeric(reviews["price"], errors="coerce")


check("reviews", reviews)
print(f"vintage extracted: {reviews.vintage_year.notna().sum():,} "
      f"({reviews.vintage_year.notna().mean()*100:.1f}%), "
      f"range {int(reviews.vintage_year.min())}-{int(reviews.vintage_year.max())}")
print(f"price missing: {int(reviews.price.isna().sum()):,} | "
      f"value_score computed on: {int(reviews.value_score.notna().sum()):,} rows")

reviews: 129,971 rows x 13 cols
columns: ['points', 'title', 'winery', 'variety', 'designation', 'country', 'province', 'region', 'sub_region', 'price', 'value_score', 'vintage_year', 'description']
vintage extracted: 123,627 (95.1%), range 1601-2017
price missing: 8,996 | value_score computed on: 120,975 rows


looks good! now to save :)

In [9]:
# (intermediate file not saved for submission -- the cleaned reviews flow
# onward in-memory; only the slim downstream CSVs are written to disk)
# reviews.to_csv(PROC / "wine_reviews_clean.csv", index=False)
print("cleaned reviews ready (intermediate file not written)")

saved: wine_reviews_clean.csv


### Clean varietal region area (2000 / 2010 / 2016 / 2023)

Combine the four grape variety by region area files from the Database of Regional, National and Global Winegrape Bearing Areas from the Wine Economics Research Centre at the University of Adelaide. **Note:** the "2023" file is really a *latest
snapshot* — its rows carry mixed vintage years (2019-2024) and some messy colour
codes. This cleaning effort will keep each row's true year and map any color that isn't R/W/G to
`unknown`.

In [10]:
frames = []
for yr in [2000, 2010, 2016, 2023]:
    d = pd.read_excel(RAW / f"variety_region_area_{yr}.xlsx")
    d.columns = [to_snake(c) for c in d.columns]
    frames.append(d)
variety = pd.concat(frames, ignore_index=True)

variety

,country,region,sub_region,sub_sub_region,prime,area,year,porigin,pcolour,psource
0,Algeria,Algeria Total,NaN,NaN,Alicante Henri Bouschet,3020.0,2000.0,France,R,RHV
1,Algeria,Algeria Total,NaN,NaN,Cabernet Sauvignon,1510.0,2000.0,France,R,RHV
2,Algeria,Algeria Total,NaN,NaN,Cinsaut,7550.0,2000.0,France,R,RHV
3,Algeria,Algeria Total,NaN,NaN,Garnacha Tinta,6040.0,2000.0,Spain,R,RHV
4,Algeria,Algeria Total,NaN,NaN,Mazuelo,7550.0,2000.0,Spain,R,RHV
...,...,...,...,...,...,...,...,...,...,...
82446,Uruguay,NaN,NaN,NaN,Vermentino,3.1,2023.0,Italy,W,RHV
82447,Uruguay,NaN,NaN,NaN,Viognier,47.5,2023.0,France,W,RHV
82448,Uruguay,NaN,NaN,NaN,other,0.1,2023.0,NaN,NaN,NaN
82449,Uruguay,NaN,NaN,NaN,other red,22.6,2023.0,NaN,R,NaN


In [11]:
# Tidy text + numeric fields
for col in ["country", "region", "prime", "pcolour"]:
    variety[col] = variety[col].astype("string").str.strip()

variety["variety_area_ha"] = pd.to_numeric(variety["area"], errors="coerce")
variety["year"] = pd.to_numeric(variety["year"], errors="coerce").astype("Int64")
variety = variety.rename(columns={"prime": "variety", "porigin": "grape_origin"})


# Map grape color code -> readable label (anything else -> unknown)
color_map = {"R": "red", "W": "white", "G": "grey_or_gris"}

variety["grape_color"] = variety["pcolour"].map(color_map).fillna("unknown")

variety = variety[["country", "region", "sub_region", "sub_sub_region",
                   "variety", "variety_area_ha", "grape_color", "year", 
                   "grape_origin"]].copy()

check("variety", variety)
print("years present:", sorted(int(y) for y in variety.year.dropna().unique()))
print("rows per source:", variety.year.value_counts().to_dict())
print("grape_color:", variety.grape_color.value_counts(dropna=False).to_dict())
print("area missing:", int(variety.variety_area_ha.isna().sum()),
      "| variety missing:", int(variety.variety.isna().sum()))


variety: 82,451 rows x 9 cols
columns: ['country', 'region', 'sub_region', 'sub_sub_region', 'variety', 'variety_area_ha', 'grape_color', 'year', 'grape_origin']
years present: [2000, 2010, 2016, 2019, 2020, 2022, 2023, 2024]
rows per source: {2010: 26404, 2000: 22338, 2016: 18625, 2022: 8749, 2023: 3236, 2020: 1771, 2024: 952, 2019: 174}
grape_color: {'red': 42395, 'white': 37365, 'grey_or_gris': 2306, 'unknown': 385}
area missing: 0 | variety missing: 0


all checks out -- now time again to save

In [12]:
variety.to_csv(PROC / "variety_region_area_clean.csv", index=False)
print("saved: variety_region_area_clean.csv")

saved: variety_region_area_clean.csv


### Parse and clean country yield tables (.txt)

This pasted from the same Adelaide/Wine-Economics text published table that has grape production field in it which is critical for my analysis but not present in the data on their webpage database -- as a result, it is the most fragile input: the data was collected in bulk over the world in 2000, 2010, and 2016 and the tables are glue together with a footnote paragraph sits inside
the 2010 block. Note as well that some rows are missing middle values and 2016 uses `na`, so this will be a treat to clean!

The following section:
- splits the three year blocks (handling the glued boundaries),
- reads each country row by anchoring the **first** value (grapevine area) and
  **third** value (winegrape area) from the left, and the **yield** and
  **production** from the right (so a missing *middle* column can't shift them),
- treats `na` as missing, exclude World/Rest-of-world/Missing/subtotal rows,
- flags any row with too few columns as `low_confidence` and leave its numbers as is

then, a validation block at the end checks parsed values against the source.

In [13]:
raw_text = (RAW / "country_yield_tables.txt").read_text(encoding="utf-8")

raw_text[:4000]

'Country-level grape/winegrape area and yield tables copied from the University of Adelaide / Wine Economics source.\n\nNotes for parsing:\n- Tables include 2000, 2010, and 2016 country-level rows.\n- Key columns to extract: country, year, total grapevine area harvested (\'000ha), total winegrape area (\'000ha), grape yield (tonnes per ha.), total grape production (kt).\n- The final table after the 2016 section is a winegrape-area change table by country.\n- Do not treat World, Rest of the world, Missing/subtotal rows as normal countries unless explicitly needed.\n\n--- RAW PASTED TABLES ---\n\n2000\nTotal\ngrapevine\narea\nharvested ª(\n\'000ha)\nShare of\nworld\ngrapevine\narea\nharvested\n(%)\nTotal winegrape area\n(\'000ha)\nShare of\nworld winegrape area\n(%)\nShare of\nwinegrapes in\ntotal\ngrapevine\narea (%)\nShare (%)\nof national\nagr. crop\narea under\ngrapevine\nGrape\nyield\n(tonnes per\nha.)\nTotal grape\nprod\'n (kt)\nShare of\nworld\ngrape\nprod\'n (%)\nAlgeria 51 0.7 3

In [14]:
# 1) Drop the winegrape-change table at the very end (not needed here).
core = raw_text.split("* These final three columns")[0]

# 2) Remove the 2010 footnote paragraph (sits between its World row and 2016 header).
core = re.sub(r"ªNon-sample.*?(?=2016\nTotal)", "", core, flags=re.S)

# 3) Un-glue year labels stuck to the previous number (e.g. "100.02010").
core = re.sub(r"(\d)(2000|2010|2016)(\nTotal\ngrapevine)", r"\1\n\2\3", core)

# 4) Split into year blocks on a line that is exactly a target year.
parts = re.split(r"\n(2000|2010|2016)\n", "\n" + core)

blocks = {parts[i]: parts[i + 1] for i in range(1, len(parts) - 1, 2)}

NUM = re.compile(r"^-?\d+(?:\.\d+)?$")            # a numeric token

AGG = {"world", "rest of the world", "old world subtotal",
       "new world subtotal", "world total"}       # rows to exclude from output

yield_rows = []

for year, block in blocks.items():
    for line in block.splitlines():
        s = line.strip()
        if not s or "Missing" in s:               # skip blanks + "Missing 9 in 2000"
            continue
        toks = s.split()
        
        # country = leading non-value tokens; values start at first number or 'na'
        i = 0
        while i < len(toks) and not (NUM.match(toks[i]) or toks[i] == "na"):
            i += 1
        country = " ".join(toks[:i]).strip().strip('"').replace("ª", "")
        values = [np.nan if v == "na" else float(v)
                  for v in toks[i:] if NUM.match(v) or v == "na"]
        if not country or len(values) < 2 or country.lower() in AGG:
            continue                              # header lines + aggregate rows
        rec = {"country": country, "year": int(year),
               "total_grapevine_area_000ha": np.nan, "total_winegrape_area_000ha": np.nan,
               "grape_yield_tonnes_per_ha": np.nan, "total_grape_production_kt": np.nan,
               "parse_confidence": "low_confidence"}
        
        if len(values) >= 8:                      # full row: trust it
            rec["total_grapevine_area_000ha"] = values[0]    # 1st value from left
            rec["total_winegrape_area_000ha"] = values[2]    # 3rd value from left
            rec["grape_yield_tonnes_per_ha"]  = values[-3]   # 3rd value from right
            rec["total_grape_production_kt"]  = values[-2]   # 2nd value from right
            rec["parse_confidence"] = "ok"
        yield_rows.append(rec)

country_yield = pd.DataFrame(yield_rows)
country_yield["total_grapevine_area_ha"] = country_yield["total_grapevine_area_000ha"] * 1000
country_yield["total_winegrape_area_ha"] = country_yield["total_winegrape_area_000ha"] * 1000

country_yield

,country,year,total_grapevine_area_000ha,total_winegrape_area_000ha,grape_yield_tonnes_per_ha,total_grape_production_kt,parse_confidence,total_grapevine_area_ha,total_winegrape_area_ha
0,Algeria,2000,51.0,30.2,3.8,193.0,ok,51000.0,30200.0
1,Argentina,2000,201.0,197.4,12.2,2461.0,ok,201000.0,197400.0
2,Armenia,2000,15.0,11.2,7.9,116.0,ok,15000.0,11200.0
3,Australia,2000,137.0,130.6,10.0,1374.0,ok,137000.0,130600.0
4,Austria,2000,48.0,48.5,6.9,332.0,ok,48000.0,48500.0
...,...,...,...,...,...,...,...,...,...
134,Turkey,2016,462.0,13.7,9.2,4000.0,ok,462000.0,13700.0
135,Ukraine,2016,50.0,25.2,8.8,378.0,ok,50000.0,25200.0
136,United Kingdom,2016,2.0,1.8,1.1,1.0,ok,2000.0,1800.0
137,United States,2016,414.0,239.6,17.1,6983.0,ok,414000.0,239600.0


In [15]:
check("country_yield", country_yield)
print("rows per year:", country_yield.year.value_counts().sort_index().to_dict())
print("parse_confidence:", country_yield.parse_confidence.value_counts().to_dict())
low = country_yield[country_yield.parse_confidence == "low_confidence"]
print("low-confidence (numbers left blank):",
      list(low[["country", "year"]].itertuples(index=False, name=None)))


country_yield: 139 rows x 9 cols
columns: ['country', 'year', 'total_grapevine_area_000ha', 'total_winegrape_area_000ha', 'grape_yield_tonnes_per_ha', 'total_grape_production_kt', 'parse_confidence', 'total_grapevine_area_ha', 'total_winegrape_area_ha']
rows per year: {2000: 38, 2010: 48, 2016: 53}
parse_confidence: {'ok': 133, 'low_confidence': 6}
low-confidence (numbers left blank): [('Taiwan', 2000), ('Myanmar', 2010), ('Taiwan', 2010), ('Cambodia', 2016), ('Myanmar', 2016), ('Norway', 2016)]


In [16]:
# everything from above checks out from visual inspection now i am going to do one more
# test to validate random rows manually
expected = {("France", 2000): (865, 864.8, 8.9, 7676),
            ("Italy", 2000):  (870, 636.7, 10.4, 9073),
            ("Israel", 2000): (7, 4.9, 14.6, 102),       # row missing a middle column
            ("China", 2010):  (547, 29.5, 15.6, 8519),
            ("United States", 2016): (414, 239.6, 17.1, 6983),
            ("Canada", 2016): (np.nan, 12.6, 9.0, 117)}  # row starts with 'na'

print("parsed vs source:")
all_ok = True

for (c, y), exp in expected.items():
    r = country_yield[(country_yield.country == c) & (country_yield.year == y)].iloc[0]
    got = (r.total_grapevine_area_000ha, r.total_winegrape_area_000ha,
           r.grape_yield_tonnes_per_ha, r.total_grape_production_kt)
    ok = all((np.isnan(a) and np.isnan(b)) or a == b for a, b in zip(got, exp))
    all_ok &= ok
    print(f"  {c} {y}: {'VALIDATED' if ok else 'MISMATCH'} -->  expected={exp}, got={got} ")
    

parsed vs source:
  France 2000: VALIDATED -->  expected=(865, 864.8, 8.9, 7676), got=(865.0, 864.8, 8.9, 7676.0) 
  Italy 2000: VALIDATED -->  expected=(870, 636.7, 10.4, 9073), got=(870.0, 636.7, 10.4, 9073.0) 
  Israel 2000: VALIDATED -->  expected=(7, 4.9, 14.6, 102), got=(7.0, 4.9, 14.6, 102.0) 
  China 2010: VALIDATED -->  expected=(547, 29.5, 15.6, 8519), got=(547.0, 29.5, 15.6, 8519.0) 
  United States 2016: VALIDATED -->  expected=(414, 239.6, 17.1, 6983), got=(414.0, 239.6, 17.1, 6983.0) 
  Canada 2016: VALIDATED -->  expected=(nan, 12.6, 9.0, 117), got=(nan, 12.6, 9.0, 117.0) 


In [17]:
country_yield = country_yield[['year', 'country', 'total_grapevine_area_ha', 'total_winegrape_area_ha','grape_yield_tonnes_per_ha', 
 'total_grape_production_kt']].copy()

country_yield.head(4)

,year,country,total_grapevine_area_ha,total_winegrape_area_ha,grape_yield_tonnes_per_ha,total_grape_production_kt
0,2000,Algeria,51000.0,30200.0,3.8,193.0
1,2000,Argentina,201000.0,197400.0,12.2,2461.0
2,2000,Armenia,15000.0,11200.0,7.9,116.0
3,2000,Australia,137000.0,130600.0,10.0,1374.0


looks good -- final save!

In [18]:
country_yield.to_csv(PROC / "country_grape_yield_clean.csv", index=False)
print("saved: country_grape_yield_clean.csv")

saved: country_grape_yield_clean.csv


## Adding Features and Measurements for data processing

### Estimate variety production

Now the fun part: putting two cleaned files together to *estimate* how much grape
each variety produces. The variety files only give **area** (hectares planted), and
the Adelaide yield table gives a country-level **yield** (tonnes per hectare) for
2000/2010/2016. So a rough estimate is:

`estimated_variety_production_tons = variety_area_ha × country grape_yield_tonnes_per_ha`

Big caveat (and i'll be honest about this in the write-ups): this assumes every
variety in a country grows at that country's *average* yield, which isn't true in
real life. It's a geography-of-grapes estimate, not a measured number. For the years
with no yield (the 2019-2024 snapshot rows) i deliberately leave production blank
instead of guessing.

In [19]:
# load the two pieces we need
variety = pd.read_csv(PROC / "variety_region_area_clean.csv", low_memory=False)
yld     = pd.read_csv(PROC / "country_grape_yield_clean.csv")

variety["year"] = pd.to_numeric(variety["year"], errors="coerce").astype("Int64")

# quick look at which years each side covers (yield only has 3 of them)
print("variety years:", sorted(variety.year.dropna().unique().tolist()))
print("yield years:  ", sorted(yld.year.unique().tolist()))
variety.head(4)

variety years: [2000, 2010, 2016, 2019, 2020, 2022, 2023, 2024]
yield years:   [2000, 2010, 2016]


,country,region,sub_region,sub_sub_region,variety,variety_area_ha,grape_color,year,grape_origin
0,Algeria,Algeria Total,NaN,NaN,Alicante Henri Bouschet,3020.0,red,2000,France
1,Algeria,Algeria Total,NaN,NaN,Cabernet Sauvignon,1510.0,red,2000,France
2,Algeria,Algeria Total,NaN,NaN,Cinsaut,7550.0,red,2000,France
3,Algeria,Algeria Total,NaN,NaN,Garnacha Tinta,6040.0,red,2000,Spain


In [20]:
# before merging i checked the country names line up across the two files.
# two things needed fixing: "Turkiye" vs "Turkey", and a junk "Missing 9" row.

variety["country"] = variety["country"].replace({"Turkiye": "Turkey"})

before = len(variety)
variety = variety[variety["country"] != "Missing 9"].copy()   # not a real country
print(f"dropped 'Missing 9' rows: {before - len(variety)}")

variety_ctry = set(variety[variety.year.isin([2000, 2010, 2016])].country.dropna())
yield_ctry = set(yld.country)
print("variety countries still missing from yield:", sorted(variety_ctry - yield_ctry)) 

dropped 'Missing 9' rows: 102
variety countries still missing from yield: []


In [21]:
# the variety file is region-level, so the same grape shows up in many regions.
# roll it up to one row per country x year x variety (sum the area).
# grape_color is per-variety, so i grab the most common label to keep it as one row.

def main_color(s):
    s = s.dropna()
    return s.mode().iloc[0] if not s.empty else "unknown"

agg = (variety.groupby(["country", "year", "variety"], as_index=False)
              .agg(variety_area_ha=("variety_area_ha", "sum"),
                   grape_color=("grape_color", main_color)))


print("country-variety-year rows:", len(agg))
agg.head(4)

country-variety-year rows: 10778


,country,year,variety,variety_area_ha,grape_color
0,Albania,2022,Debine e Bardhë,71.0,white
1,Albania,2022,Kadarka,111.0,red
2,Albania,2022,Merlot,735.0,red
3,Albania,2022,Pules,150.0,white


In [22]:
# merge on country+year so each variety row picks up its country-year yield.
# yield only exists for 2000/2010/2016 -> every other year stays NaN (no estimate).
est = agg.merge(yld[["country", "year", "grape_yield_tonnes_per_ha"]],
                on=["country", "year"], how="left")

# only multiply where we actually have BOTH numbers. otherwise leave it blank.
est["estimated_variety_production_tons"] = np.where(
    est["grape_yield_tonnes_per_ha"].notna() & est["variety_area_ha"].notna(),
    est["variety_area_ha"] * est["grape_yield_tonnes_per_ha"], np.nan)

check("est", est)
print("rows WITH an estimate:   ", int(est.estimated_variety_production_tons.notna().sum()))
print("rows WITHOUT (no yield): ", int(est.estimated_variety_production_tons.isna().sum()))
print("estimate present by year:",
      est[est.estimated_variety_production_tons.notna()].year.value_counts().sort_index().to_dict())
print("estimate ABSENT by year: ",
      est[est.estimated_variety_production_tons.isna()].year.value_counts().sort_index().to_dict())

est: 10,778 rows x 7 cols
columns: ['country', 'year', 'variety', 'variety_area_ha', 'grape_color', 'grape_yield_tonnes_per_ha', 'estimated_variety_production_tons']
rows WITH an estimate:    7561
rows WITHOUT (no yield):  3217
estimate present by year: {2000: 1699, 2010: 2669, 2016: 3193}
estimate ABSENT by year:  {2000: 4, 2010: 15, 2016: 15, 2019: 170, 2020: 332, 2022: 1736, 2023: 734, 2024: 211}


In [23]:
# manual spot-check so i trust the multiplication: France 2016 Merlot
r = est[(est.country == "France") & (est.year == 2016) & (est.variety == "Merlot")].iloc[0]
print(f"France 2016 Merlot: {r.variety_area_ha:,.0f} ha x {r.grape_yield_tonnes_per_ha} t/ha "
      f"= {r.estimated_variety_production_tons:,.0f} t")
print("matches by hand:", abs(r.variety_area_ha * r.grape_yield_tonnes_per_ha
                              - r.estimated_variety_production_tons) < 1)

# and a sanity peek at the biggest grapes in 2016 -- should look like the usual suspects
print("\nTop varieties by estimated production, 2016:")
print(est[est.year == 2016].groupby("variety")["estimated_variety_production_tons"]
      .sum().sort_values(ascending=False).head(8).round(0).to_string())

France 2016 Merlot: 108,483 ha x 7.7 t/ha = 835,318 t
matches by hand: True

Top varieties by estimated production, 2016:
variety
Cabernet Sauvignon    3483574.0
Merlot                2555211.0
Chardonnay            2286913.0
other red             2087262.0
Syrah                 1846650.0
Tempranillo           1410997.0
Airén                 1325336.0
Sauvignon Blanc       1240591.0


looks right -- Cabernet/Merlot/Chardonnay on top is what i'd expect from the investigation of the review data aggregates as a proxy for popularity and thus production <--> and vice versa

In [24]:
est.to_csv(PROC / "variety_country_year_estimates.csv", index=False)
print("saved: variety_country_year_estimates.csv")

saved: variety_country_year_estimates.csv


### Wine style / color for reviews

The review data has **no color or style field**, so i build a practical
`wine_style` from the variety name + the wine's title. Categories: red, white,
rose, sparkling, other. Note this is *not* the same as the grape_color in the
variety files -- a style is about the finished wine (a red grape can become a rosé
or a sparkling), so i keep them separate and don't mix the two up.

Order of the rules matters: check rosé first, then sparkling, then known red/white
variety lists, then a couple of fuzzy "blanc/bianco/red" catches. Anything genuinely
ambiguous (Port, Sherry, odd blends) stays `other` .

In [25]:
import unicodedata

# strip accents so "Carmenere"/"Gewurztraminer"/"Albarino" match my plain-text lists
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

# The review data has no color/style field, so i build a practical wine_style from the
# variety name + the wine's title. I match on TOKENS (substrings) not whole names, so
# variants like "Johannisberg Riesling" or "Cabernet Sauvignon-Merlot" classify correctly.
# Order/precedence matters a lot -- see the function below.
SPARKLING = ["sparkling", "champagne", "prosecco", "cava", "spumante", "cremant", "sekt",
             "franciacorta", "blanc de blancs", "blanc de noirs", "metodo classico"]

# an explicit "this is the WHITE version" marker -> white wins even over a red grape name
# (so "Grenache Blanc" and "Pinot Gris" come out white, not red)
WHITE_MARK = ["blanc", "bianco", "branco", " gris", "grigio", " white"]

# specific RED grape tokens (bare "sauvignon" is deliberately NOT here, because it lives
# inside "Cabernet Sauvignon" (red) -- the white "Sauvignon Blanc" is caught by WHITE_MARK)
RED_TERMS = {"noir", "cabernet", "merlot", "syrah", "shiraz", "zinfandel", "malbec",
             "sangiovese", "tempranillo", "grenache", "garnacha", "nebbiolo", "sirah",
             "barbera", "mourvedre", "carmenere", "verdot", "gamay", "montepulciano",
             "primitivo", "aglianico", "tannat", "carignan", "dolcetto", "pinotage",
             "touriga", "blaufrankisch", "lemberger", "mencia", "monastrell", "negroamaro",
             "bonarda", "zweigelt", "nerello", "sagrantino", "schiava", "lagrein",
             "spatburgunder", "tinto", "tinta", "nero", "rosso", "rouge", " red", "corvina",
             "agiorgitiko", "saperavi", "baga", "castelao", "trincadeira", "alfrocheiro",
             "bobal", "graciano", "mazuelo", "frappato", "refosco", "teroldego", "dornfelder",
             "meritage", "g-s-m", "cinsault", "petite sirah", "st. laurent", "bouschet",
             "lambrusco", "alicante"}

# specific WHITE grape tokens
WHITE_TERMS = {"chardonnay", "riesling", "gewurztraminer", "viognier", "chenin", "semillon",
               "albarino", "alvarinho", "gruner", "veltliner", "moscat", "muscat", "vermentino",
               "gavi", "melon", "marsanne", "roussanne", "torrontes", "verdejo", "glera",
               "garganega", "fiano", "greco", "vidal", "cortese", "trebbiano", "muscadet",
               "grillo", "verdicchio", "silvaner", "kerner", "thurgau", "furmint", "assyrtiko",
               "godello", "viura", "colombard", "sylvaner", "arneis", "arinto", "verdelho",
               "vernaccia", "falanghina", "ribolla", "friulano", "malvasia", "welschriesling",
               "roditis", "moschofilero", "savatiano", "airen", "macabeo", "xarel", "parellada",
               "pecorino", "passerina", "grechetto", "carricante", "catarratto", "insolia",
               "zibibbo", "encruzado", "loureiro", "avesso", "rabigato", "turbiana", "manseng",
               "fernao pires", "palomino"}

def wine_style(variety, title):
    v = strip_accents(str(variety).lower().strip()) if isinstance(variety, str) else ""
    t = strip_accents(str(title).lower()) if isinstance(title, str) else ""
    blob = v + " " + t
    vpad = " " + v + " "                      # pad so " red"/" gris" match at word edges
    if "rose" in v or "rosato" in v or "rosado" in v:    # rosé first
        return "rose"
    if any(k in blob for k in SPARKLING):                # then sparkling
        return "sparkling"
    if v == "sauvignon":                                 # bare "Sauvignon" -> Sauvignon Blanc
        return "white"
    if any(m in vpad for m in WHITE_MARK):               # explicit white-version marker wins
        return "white"
    has_red = any(term in vpad for term in RED_TERMS)
    has_white = any(term in vpad for term in WHITE_TERMS)
    if has_red and not has_white:
        return "red"
    if has_white and not has_red:
        return "white"
    return "other"                                       # fortified / mixed / genuinely unclear

In [26]:
reviews = pd.read_csv(PROC / "wine_reviews_clean.csv", low_memory=False)

reviews["wine_style"] = [wine_style(v, t) for v, t in zip(reviews["variety"], reviews["title"])]

print("wine_style counts:")
print(reviews.wine_style.value_counts(dropna=False).to_string())
covered = reviews.wine_style.isin(["red", "white", "rose", "sparkling"]).mean()
print(f"\nclassified into a clear style: {covered*100:.1f}%")
print("\nwhat stays 'other' (top 10):")
reviews[reviews.wine_style == "other"].variety.value_counts().head(10)

wine_style counts:
wine_style
red          77542
white        40406
sparkling     5378
rose          3998
other         2647

classified into a clear style: 98.0%

what stays 'other' (top 10):


variety
Port                  668
Sherry                100
Weissburgunder         47
Prugnolo Gentile       43
Assyrtico              43
Pedro Ximénez          42
Traminer               39
Shiraz-Viognier        38
Gelber Muskateller     36
Rkatsiteli             33
Name: count, dtype: int64

98% land in a clear style now. I match on tokens, not whole variety names, so "Johannisberg Riesling" reads as white and "Cabernet Sauvignon" stays red (bare "sauvignon" is kept out of the white list on purpose). The leftovers are genuinely ambiguous -- Port, Sherry, odd blends -- so they honestly stay "other". saving.

In [27]:
# (intermediate file not saved for submission -- the cleaned reviews flow
# onward in-memory; only the slim downstream CSVs are written to disk)
# reviews.to_csv(PROC / "wine_reviews_with_style.csv", index=False)
print("styled reviews ready (intermediate file not written)")

saved: wine_reviews_with_style.csv


#### adding old versus new world feature to countries

In [28]:
# load the cleaned + derived pieces we'll reshape for Tableau
oiv      = pd.read_csv(PROC / "oiv_cleaned.csv")
reviews  = pd.read_csv(PROC / "wine_reviews_with_style.csv", low_memory=False)
est      = pd.read_csv(PROC / "variety_country_year_estimates.csv")

# old/new world map has no header and uses OW/NW codes -> turn into readable labels
onw = pd.read_excel(RAW / "old_new_world_mapping.xlsx", header=None,
                    names=["country", "world_code"])
onw["world"] = onw["world_code"].map({"OW": "Old World", "NW": "New World"})

# small alias map so the OIV/review names line up with the world map
NAME_FIX = {"United States of America": "United States"}
for df in (oiv, reviews, est):
    df["country"] = df["country"].replace(NAME_FIX)

# OIV has a "Global" world-total row -- drop it so it can't dominate the map/bars
oiv = oiv[oiv["country"] != "Global"].copy()

print("old/new world:", onw.world.value_counts().to_dict())
print("OIV countries after dropping Global:", oiv.country.nunique())

old/new world: {'Old World': 31, 'New World': 22}
OIV countries after dropping Global: 219


#### oiv_yearly.csv : 10 year moving average production & consumption per country 


In [29]:
oiv_yearly = oiv_cleaned.sort_values(["country", "year"]).copy()

# 10-year moving averages (plain hl)
oiv_yearly["production_10yr_ma_hl"] = (
    oiv_yearly.groupby("country")["production_hl"]
    .transform(lambda s: s.rolling(10, min_periods=3).mean())
)

oiv_yearly["consumption_10yr_ma_hl"] = (
    oiv_yearly.groupby("country")["consumption_hl"]
    .transform(lambda s: s.rolling(10, min_periods=3).mean())
)

# add Old / New World tag
oiv_yearly = oiv_yearly.merge(
    onw[["country", "world"]], on="country", how="left")

check("tableau_oiv_yearly", oiv_yearly)
print("countries tagged Old/New World:", int(oiv_yearly.world.notna().sum()), "/", len(oiv_yearly))
print("\nTop 5 producers:")
oiv_yearly.nlargest(5, "production_10yr_ma_hl")

tableau_oiv_yearly: 6,591 rows x 7 cols
columns: ['country', 'year', 'consumption_hl', 'production_hl', 'production_10yr_ma_hl', 'consumption_10yr_ma_hl', 'world']
countries tagged Old/New World: 1372 / 6591

Top 5 producers:


,country,year,consumption_hl,production_hl,production_10yr_ma_hl,consumption_10yr_ma_hl,world
2335,Global,2013,246235000.0,294022000.0,275404200.0,243129000.0,NaN
2337,Global,2015,241140000.0,278620000.0,273255000.0,243441500.0,NaN
2330,Global,2008,246274000.0,270078000.0,273184800.0,236676800.0,NaN
2336,Global,2014,240679000.0,271820000.0,273043000.0,243350100.0,NaN
2340,Global,2018,241912000.0,296129000.0,272852700.0,242885100.0,NaN


In [30]:
oiv_yearly.to_csv(PROC / "oiv_yearly.csv", index=False)

#### oiv_country_summary : country summary file

In [31]:
# country-level summary. NOTE: values are in plain hl (columns named *_hl accordingly).
oiv_country_summary = (
    oiv_cleaned.sort_values(["country", "year"])
    .groupby("country", as_index=False)
    .agg(
        avg_production_hl=("production_hl", "mean"),
        avg_consumption_hl=("consumption_hl", "mean"),
        latest_production_hl=("production_hl", "last"),
        latest_consumption_hl=("consumption_hl", "last"),
        n_years=("year", "nunique"),
        latest_year=("year", "max"),
    )
)

oiv_country_summary = oiv_country_summary.merge(
    onw[["country", "world"]], on="country", how="left")

check("tableau_oiv_country_summary", oiv_country_summary)
print("\nTop 5 producers:")
oiv_country_summary.nlargest(5, "avg_production_hl")

tableau_oiv_country_summary: 220 rows x 8 cols
columns: ['country', 'avg_production_hl', 'avg_consumption_hl', 'latest_production_hl', 'latest_consumption_hl', 'n_years', 'latest_year', 'world']

Top 5 producers:


,country,avg_production_hl,avg_consumption_hl,latest_production_hl,latest_consumption_hl,n_years,latest_year,world
77,Global,2.658329e+08,2.344204e+08,226706000.0,208000000.0,31,2025,NaN
96,Italy,4.875468e+07,2.590741e+07,44384000.0,20195000.0,32,2025,Old World
69,France,4.768300e+07,2.982494e+07,36079000.0,21965000.0,31,2025,Old World
188,Spain,3.539748e+07,1.171700e+07,28679000.0,9356000.0,31,2025,Old World
210,United States of America,2.244581e+07,2.830155e+07,20000000.0,31909000.0,31,2025,NaN


In [32]:
# saving
oiv_country_summary.to_csv(PROC / "oiv_country_summary.csv",index=False)

#### variety_production.csv : country x variety x year estimates

In [33]:
var_prod = est[["country", "year", "variety", "grape_color", "variety_area_ha",
                "grape_yield_tonnes_per_ha", "estimated_variety_production_tons"]].copy()

var_prod = var_prod.merge(onw[["country", "world"]], on="country", how="left")

check("variety_production", var_prod)
print("rows with an estimate:", int(var_prod.estimated_variety_production_tons.notna().sum()),"/", len(var_prod))

variety_production: 10,778 rows x 8 cols
columns: ['country', 'year', 'variety', 'grape_color', 'variety_area_ha', 'grape_yield_tonnes_per_ha', 'estimated_variety_production_tons', 'world']
rows with an estimate: 7561 / 10778


In [34]:
var_prod.to_csv(PROC / "variety_production.csv", index=False)

#### reviews.csv : review-level, trimmed, with wine_style 

In [35]:
rev_cols = ["title", "winery", "country", "province", "variety", "wine_style",
            "points", "price", "value_score", "vintage_year"]

rev = reviews[[c for c in rev_cols if c in reviews.columns]].copy()
rev = rev.merge(onw[["country", "world"]], on="country", how="left")

check("reviews", rev)
print("review rows tagged with a world:", f"{rev.world.notna().mean()*100:.1f}%")

rev.to_csv(PROC / "reviews.csv", index=False)
print("saved: reviews.csv")


reviews: 129,971 rows x 11 cols
columns: ['title', 'winery', 'country', 'province', 'variety', 'wine_style', 'points', 'price', 'value_score', 'vintage_year', 'world']
review rows tagged with a world: 57.9%
saved: reviews.csv


#### variety_region.csv : region (and sub-region) x variety x year

Adding a regional view so the dashboard can drill **country → region → sub-region**.
A few honest notes on what the data actually supports:
- `region` is filled on ~98% of rows, so the region view is solid.
- `sub_region` is only ~8% (mostly Argentina), so that drilldown is real but sparse.
- `sub_sub_region` is **completely empty** in the cleaned file, so i'm *not* making a
  sub-sub view -- there's nothing to map and i won't invent it.

Two cleanups first: the source has `"<Country> Total"` rollup rows mixed in with real
regions (35 countries have both!), so i pull those out so area never double-counts; and
a handful of region names have stray newlines baked in (e.g. South African regions), so
i squash whitespace before grouping.

In [36]:
# region names have some embedded newlines / doubled spaces from the source -> tidy them
def tidy(s):
    if pd.isna(s):
        return s
    return re.sub(r"\s+", " ", str(s)).strip()

for col in ["region", "sub_region"]:
    variety[col] = variety[col].map(tidy)

# the "<Country> Total" rows are country rollups, not real sub-national regions.
# splitting them off so summing region area can't double-count a country.
is_total = variety["region"].astype("string").str.endswith("Total").fillna(False)
regions = variety[~is_total].copy()
print("rows total:", len(variety),
      "| '... Total' rollup rows pulled out:", int(is_total.sum()),
      "| real region rows:", len(regions))

rows total: 82349 | '... Total' rollup rows pulled out: 1271 | real region rows: 81078


In [37]:
# build region x variety x year, summing area (same region+variety can span sub_region lines)
reg = (regions.groupby(["country", "region", "sub_region", "year", "variety", "grape_color"],
                       as_index=False, dropna=False)
              .agg(variety_area_ha=("variety_area_ha", "sum")))

# same estimate idea as Section 6: area x country-year yield, only where yield exists
reg = reg.merge(yld[["country", "year", "grape_yield_tonnes_per_ha"]],
                on=["country", "year"], how="left")
reg["estimated_variety_production_tons"] = np.where(
    reg.grape_yield_tonnes_per_ha.notna() & reg.variety_area_ha.notna(),
    reg.variety_area_ha * reg.grape_yield_tonnes_per_ha, np.nan)

reg = reg.merge(onw[["country", "world"]], on="country", how="left")

# a little flag so Tableau knows whether a row is region-level or drills to a sub_region
reg["geo_level"] = np.where(reg.sub_region.notna(), "sub_region", "region")

check("variety_region", reg)
print("rows with an estimate:", int(reg.estimated_variety_production_tons.notna().sum()), "/", len(reg))
print("geo_level:", reg.geo_level.value_counts().to_dict())
print("distinct regions:", reg.region.nunique(), "| distinct sub_regions:", reg.sub_region.nunique())
print("any region names still holding a newline?:", int(reg.region.astype(str).str.contains("\n").sum()))

variety_region: 81,039 rows x 11 cols
columns: ['country', 'region', 'sub_region', 'year', 'variety', 'grape_color', 'variety_area_ha', 'grape_yield_tonnes_per_ha', 'estimated_variety_production_tons', 'world', 'geo_level']
rows with an estimate: 65994 / 81039
geo_level: {'region': 74495, 'sub_region': 6544}
distinct regions: 1117 | distinct sub_regions: 285
any region names still holding a newline?: 0


In [38]:
reg.to_csv(PROC / "variety_region.csv", index=False)
print("saved: variety_region.csv")

saved: variety_region.csv


#### region_summary.csv : one row per region

A tidy per-region rollup for quick maps and ranking: total planted area, estimated
production, how many varieties, the single biggest variety, and the dominant grape
color -- taken from each region's **latest** available year so i'm not mixing 2000 and
2023 areas in one number.

(Heads-up i noticed: a couple of Spanish/French regions show up twice under slightly
different spellings -- "Castilla La Mancha" vs "Castilla-La Mancha". Those are genuinely
different source records from different years, so i leave them separate rather than merge
them and invent a combined total.)

In [39]:
# take each region's latest year so areas aren't summed across different snapshots
latest = reg.groupby(["country", "region"], as_index=False).agg(latest_year=("year", "max"))
reg_latest = reg.merge(latest, on=["country", "region"]).query("year == latest_year")

# main rollup numbers per region
region_summary = (reg_latest.groupby(["country", "region", "world"], as_index=False, dropna=False)
                  .agg(total_area_ha=("variety_area_ha", "sum"),
                       est_production_tons=("estimated_variety_production_tons", "sum"),
                       n_varieties=("variety", "nunique"),
                       latest_year=("year", "max")))

# biggest variety by area in each region
top_variety = (reg_latest.groupby(["country", "region"])
               .apply(lambda d: d.groupby("variety")["variety_area_ha"].sum().idxmax())
               .rename("top_variety").reset_index())

# dominant grape colour by area in each region
dom_color = (reg_latest.groupby(["country", "region", "grape_color"])["variety_area_ha"].sum()
             .reset_index().sort_values("variety_area_ha", ascending=False)
             .drop_duplicates(["country", "region"])[["country", "region", "grape_color"]]
             .rename(columns={"grape_color": "dominant_color"}))

region_summary = (region_summary
                  .merge(top_variety, on=["country", "region"], how="left")
                  .merge(dom_color, on=["country", "region"], how="left"))

check("region_summary", region_summary)
print("\nTop 6 regions by planted area:")
region_summary.nlargest(6, "total_area_ha")[["country", "region", "total_area_ha",
                                             "top_variety", "dominant_color", "world"]]

C:\Users\jtrob\AppData\Local\Temp\ipykernel_10504\689695776.py:3: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  reg_latest = reg.merge(latest, on=["country", "region"]).query("year == latest_year")


region_summary: 1,131 rows x 9 cols
columns: ['country', 'region', 'world', 'total_area_ha', 'est_production_tons', 'n_varieties', 'latest_year', 'top_variety', 'dominant_color']

Top 6 regions by planted area:


,country,region,total_area_ha,top_variety,dominant_color,world
897,Spain,Castilla La Mancha,442971.000000,Airén,white,Old World
900,Spain,Castilla-La Mancha,409968.942900,Airén,white,Old World
412,France,Languedoc Roussillon,245964.278500,Syrah,red,Old World
1005,United States,California,181794.261405,Cabernet Sauvignon,red,New World
902,Spain,Ciudad Real,175764.000000,Airén,white,Old World
413,France,Languedoc-Roussillon,173032.608700,Syrah,red,Old World


In [40]:
region_summary.to_csv(PROC / "region_summary.csv", index=False)
print("saved: region_summary.csv")

saved: region_summary.csv


all the Tableau files are written (now six -- the four country-level ones plus the two new regional ones) and the row counts look sensible. on to the poster + HTML next!

## Inkscape Poster files

Four small, focused CSVs for the consumer SVG poster ("How to Pick a Bottle Without
Pretending You Know Wine"). Everything here comes from the reviewed-wine data, so the
poster can stay honest -- it's *critic coverage*, not the whole market.

- `poster_summary_stats.csv` -- box-plot five-number summaries (points and value_score) per wine style
- `poster_top_quality.csv` -- the highest-scoring reviewed wines
- `poster_top_value.csv` -- best points-per-dollar wines (with a quality floor)
- `poster_quality_indicators.csv` -- groups that tend to score above the overall average

In [41]:
reviews = pd.read_csv(PROC / "wine_reviews_with_style.csv", low_memory=False)
STYLES = ["red", "white", "rose", "sparkling", "other"]
overall_avg_points = reviews["points"].mean()
print("overall average points:", round(overall_avg_points, 2))

overall average points: 88.45


#### poster_summary_stats.csv

In [42]:
# points uses every review; value_score only uses rows with a positive price.

rows = []
for metric, src in [("points", reviews), ("value_score", reviews[reviews["price"] > 0])]:
    for st in STYLES:
        s = src.loc[src["wine_style"] == st, metric].dropna()
        if s.empty:
            continue
        rows.append(dict(wine_style=st, metric=metric, n=int(len(s)),
                         min=round(s.min(), 2), q1=round(s.quantile(.25), 2),
                         median=round(s.median(), 2), q3=round(s.quantile(.75), 2),
                         max=round(s.max(), 2)))
poster_summary = pd.DataFrame(rows)
check("poster_summary_stats", poster_summary)
poster_summary

poster_summary_stats: 10 rows x 8 cols
columns: ['wine_style', 'metric', 'n', 'min', 'q1', 'median', 'q3', 'max']


,wine_style,metric,n,min,q1,median,q3,max
0,red,points,77542,80.00,87.00,89.00,91.00,100.00
1,white,points,40406,80.00,86.00,88.00,90.00,100.00
2,rose,points,3998,80.00,85.00,87.00,88.00,96.00
3,sparkling,points,5378,80.00,87.00,88.00,91.00,100.00
4,other,points,2647,80.00,87.00,88.00,90.00,100.00
5,red,value_score,72570,0.03,1.84,2.90,4.53,21.50
6,white,value_score,37624,0.05,2.97,4.32,5.80,21.50
7,rose,value_score,3636,0.11,4.25,5.29,6.62,16.80
8,sparkling,value_score,4833,0.12,1.74,2.93,4.58,17.40
9,other,value_score,2312,0.10,2.30,3.60,4.89,12.43


In [43]:
poster_summary.to_csv(PROC / "poster_summary_stats.csv", index=False)

#### poster_top_quality.csv : highest-scoring reviewed wines (top 5 distinct)

In [44]:
cols = ["title", "country", "variety", "wine_style", "points", "price", "value_score"]
top_quality = (reviews.sort_values(["points", "value_score"], ascending=[False, False])
                      .drop_duplicates("title").head(5)[cols])

check("poster_top_quality", top_quality)
top_quality

poster_top_quality: 5 rows x 7 cols
columns: ['title', 'country', 'variety', 'wine_style', 'points', 'price', 'value_score']


,title,country,variety,wine_style,points,price,value_score
113929,Charles Smith 2006 Royal City Syrah (Columbia ...,US,Syrah,red,100,80.0,1.250000
123545,Cayuse 2008 Bionic Frog Syrah (Walla Walla Val...,US,Syrah,red,100,80.0,1.250000
58352,Château Léoville Barton 2010 Saint-Julien,France,Bordeaux-style Red Blend,red,100,150.0,0.666667
45798,Cardinale 2006 Cabernet Sauvignon (Napa Valley),US,Cabernet Sauvignon,red,100,200.0,0.500000
7335,Avignonesi 1995 Occhio di Pernice (Vin Santo ...,Italy,Prugnolo Gentile,other,100,210.0,0.476190


In [45]:
top_quality.to_csv(PROC / "poster_top_quality.csv", index=False)


#### poster_top_value.csv : best points-per-dollar, with a quality floor

In [46]:
top_value = (reviews[(reviews["points"] >= 88) & (reviews["price"] > 0)
                     & reviews["value_score"].notna()]
             .sort_values("value_score", ascending=False)
             .drop_duplicates("title").head(5)[cols])

check("poster_top_value", top_value)
top_value

poster_top_value: 5 rows x 7 cols
columns: ['title', 'country', 'variety', 'wine_style', 'points', 'price', 'value_score']


,title,country,variety,wine_style,points,price,value_score
119116,Landshut 2013 Late Harvest Spätlese Riesling (...,Germany,Riesling,white,88,6.0,14.666667
44054,Ste. Chapelle 2001 Johannisberg Riesling (Idaho),US,Johannisberg Riesling,white,88,6.0,14.666667
65340,Ste. Chapelle 2001 Dry Gewürztraminer (Idaho),US,Gewürztraminer,white,88,6.0,14.666667
114104,Santa Ines 2000 Legado de Armida Reserva Chard...,Chile,Chardonnay,white,88,6.0,14.666667
44065,Ste. Chapelle 2001 Dry Riesling (Idaho),US,Riesling,white,88,6.0,14.666667


In [47]:
top_value.to_csv(PROC / "poster_top_value.csv", index=False)

#### poster_quality_indicators.csv : groups scoring above the overall average 

In [48]:
# look at countries and varieties with enough reviews to be meaningful (n >= 200),
# keep those above average, and take the 10 with the biggest "lift" over the mean.
def indicators(col, label, min_n=200):
    g = reviews.groupby(col).agg(n=("points", "size"), avg_points=("points", "mean"))
    g = g[g["n"] >= min_n].copy()
    g["lift"] = g["avg_points"] - overall_avg_points
    g["dimension"] = label
    return g.reset_index().rename(columns={col: "category"})[
        ["dimension", "category", "n", "avg_points", "lift"]]

quality_indicators = pd.concat([indicators("country", "Country"),
                                indicators("variety", "Variety")], ignore_index=True)
quality_indicators = (quality_indicators[quality_indicators["lift"] > 0]
                      .sort_values("lift", ascending=False).head(10))
quality_indicators[["avg_points", "lift"]] = quality_indicators[["avg_points", "lift"]].round(2)

check("poster_quality_indicators", quality_indicators)
quality_indicators

poster_quality_indicators: 10 rows x 5 cols
columns: ['dimension', 'category', 'n', 'avg_points', 'lift']


,dimension,category,n,avg_points,lift
61,Variety,Sangiovese Grosso,751,90.53,2.08
42,Variety,Nebbiolo,2804,90.25,1.80
2,Country,Austria,3345,90.10,1.65
18,Variety,Blaufränkisch,232,90.02,1.57
34,Variety,Grüner Veltliner,1345,89.98,1.53
6,Country,Germany,2165,89.85,1.40
50,Variety,Port,668,89.73,1.29
24,Variety,Champagne Blend,1396,89.66,1.22
58,Variety,Riesling,5189,89.45,1.00
49,Variety,Pinot Noir,13272,89.41,0.96


In [49]:
quality_indicators.to_csv(PROC / "poster_quality_indicators.csv", index=False)

these csv files are for the inkscape poster visualizations.

### one master map file for the Tableau map view

For Tableau i just want ONE map, not a pile of charts. so i fold everything geographic into a
single long file where every row maps at the country level (regions just ride along inside their
country), and `metric_type` / `map_level` / `variety` / `grape_color` do the slicing.

`tableau/tableau_global_wine_map_master.csv` stacks up these row types:
- country wine **production** (hl) and **consumption** (hl) from OIV
- **grape area** (hectares) at the region level
- **estimated grape production** (tons) at BOTH country and region level
- a wine-**equivalent** of that estimate (hl) so grapes can sit on the same volume footing as wine

the region-level estimate uses the country's grape yield applied to each region's area (same
assumption as the country estimate) -- it rolls up exactly to the country totals, so i'm not
inventing anything new, just splitting the same number across regions. looks good, saving :)

> heads up for Tableau: estimated production now exists at BOTH country and region grain, and they roll up to the exact same totals (France 2016 = 6,274,592 t either way). so in the dashboard `map_level` has to be a **single-value** filter -- pick ONE map_level at a time, otherwise the map adds country + region rows together and doubles the number.

In [50]:
oiv  = pd.read_csv(PROC / "oiv_cleaned.csv")                            
vreg = pd.read_csv(PROC / "variety_region_area_clean.csv", low_memory=False)  
vreg_est = pd.read_csv(PROC / "variety_region.csv", low_memory=False)
est  = pd.read_csv(PROC / "variety_country_year_estimates.csv")          # country estimated production (tons)

# old / new world tag
onw = pd.read_excel(RAW / "old_new_world_mapping.xlsx", header=None,
                    names=["country", "world_code"])
onw["world"] = onw["world_code"].map({"OW": "Old World", "NW": "New World"})
onw = onw[["country", "world"]]

# nudge a couple of country spellings so they line up with the world map
oiv["country"] = oiv["country"].replace({"United States of America": "United States"})

# 10-year moving averages on the OIV series (nice optional smoothing fields)
oiv = oiv.sort_values(["country", "year"])
oiv["production_10yr_ma_hl"] = (oiv.groupby("country")["production_hl"]
    .transform(lambda s: s.rolling(10, min_periods=1).mean()))
oiv["consumption_10yr_ma_hl"] = (oiv.groupby("country")["consumption_hl"]
    .transform(lambda s: s.rolling(10, min_periods=1).mean()))

# little helper so every block comes out with the exact same columns
def block(df, map_level, metric_type, value_col, unit,
          region="All", variety="All", grape_color="All",
          prod_ma=False, cons_ma=False):
    d = df.dropna(subset=[value_col]).copy()
    return pd.DataFrame({
        "map_level": map_level,
        "metric_type": metric_type,
        "geography_name": d["country"],          # ALWAYS the country, so the whole file maps cleanly
        "country": d["country"],
        "region": (d[region] if region in d.columns else region),
        "year": d["year"],
        "variety": (d[variety] if variety in d.columns else variety),
        "grape_color": (d[grape_color].fillna("unknown") if grape_color in d.columns else grape_color),
        "map_value": d[value_col],
        "map_unit": unit,
        "production_10yr_ma_hl": d["production_10yr_ma_hl"] if prod_ma else np.nan,
        "consumption_10yr_ma_hl": d["consumption_10yr_ma_hl"] if cons_ma else np.nan,
    })

# wine-equivalent factor: 1 ton of grapes ~= 5.7 hl of wine
# (Wine Spectator / "Ask Dr. Vinny": ~150 gallons per ton -> 5.68 hl, rounded). it's a rough,
# clearly-labelled conversion -- NOT measured wine -- so i tag its unit "hl (est.)".
HL_PER_TON = 5.7

# build the region estimate frames once (reuse for tons + wine-equivalent)
vreg_est_ok = vreg_est.dropna(subset=["estimated_variety_production_tons"]).copy()
vreg_est_ok["wine_equiv_hl"] = vreg_est_ok["estimated_variety_production_tons"] * HL_PER_TON
# and the country ones
est_ok = est.dropna(subset=["estimated_variety_production_tons"]).copy()
est_ok["wine_equiv_hl"] = est_ok["estimated_variety_production_tons"] * HL_PER_TON

blocks = [
    # OIV wine volume, country level
    block(oiv, "Country", "Wine production",  "production_hl",  "hl", prod_ma=True),
    block(oiv, "Country", "Wine consumption", "consumption_hl", "hl", cons_ma=True),

    # grape area, region level
    block(vreg.assign(region=vreg["region"].fillna("All")),
          "Region", "Grape area", "variety_area_ha", "hectares",
          region="region", variety="variety", grape_color="grape_color"),

    # estimated grape production (tons) -- country AND region
    block(est_ok, "Country", "Estimated grape production", "estimated_variety_production_tons",
          "tons", variety="variety", grape_color="grape_color"),
    block(vreg_est_ok.assign(region=vreg_est_ok["region"].fillna("All")),
          "Region", "Estimated grape production", "estimated_variety_production_tons",
          "tons", region="region", variety="variety", grape_color="grape_color"),

    # wine-equivalent of that estimate (hl) -- country AND region
    block(est_ok, "Country", "Estimated grape production (wine-equivalent)",
          "wine_equiv_hl", "hl (est.)", variety="variety", grape_color="grape_color"),
    block(vreg_est_ok.assign(region=vreg_est_ok["region"].fillna("All")),
          "Region", "Estimated grape production (wine-equivalent)",
          "wine_equiv_hl", "hl (est.)", region="region", variety="variety", grape_color="grape_color"),
]

master = pd.concat(blocks, ignore_index=True)

# attach old/new world, then drop anything we can't classify so the filter has no Null
master = master.merge(onw, on="country", how="left")
n_null = master["world"].isna().sum()
master = master.dropna(subset=["world"]).reset_index(drop=True)

master.head(4)

,map_level,metric_type,geography_name,country,region,year,variety,grape_color,map_value,map_unit,production_10yr_ma_hl,consumption_10yr_ma_hl,world
0,Country,Wine production,Algeria,Algeria,All,1995.0,All,All,571000.0,hl,571000.0,NaN,Old World
1,Country,Wine production,Algeria,Algeria,All,1996.0,All,All,392000.0,hl,481500.0,NaN,Old World
2,Country,Wine production,Algeria,Algeria,All,1997.0,All,All,357000.0,hl,440000.0,NaN,Old World
3,Country,Wine production,Algeria,Algeria,All,1998.0,All,All,360000.0,hl,420000.0,NaN,Old World


In [51]:
# checking that this master file has everything i want

print("rows:", len(master))
print("columns:", list(master.columns))
print("\ncount by map_level x metric_type:")
print(master.groupby(["map_level", "metric_type"]).size())
print("\nunique metric_type:", sorted(master["metric_type"].unique()))
print("unique world:", sorted(master["world"].unique()))
print("\ntop countries by map_value, per metric_type:")
for mt in master["metric_type"].unique():
    top = (master[master["metric_type"] == mt]
           .groupby("country")["map_value"].sum().nlargest(3))
    print(f"  {mt}: " + ", ".join(f"{c} ({v:,.0f})" for c, v in top.items()))

rows: 230826
columns: ['map_level', 'metric_type', 'geography_name', 'country', 'region', 'year', 'variety', 'grape_color', 'map_value', 'map_unit', 'production_10yr_ma_hl', 'consumption_10yr_ma_hl', 'world']

count by map_level x metric_type:
map_level  metric_type                                 
Country    Estimated grape production                       7561
           Estimated grape production (wine-equivalent)     7561
           Wine consumption                                 1400
           Wine production                                  1370
Region     Estimated grape production                      65994
           Estimated grape production (wine-equivalent)    65994
           Grape area                                      80946
dtype: int64

unique metric_type: ['Estimated grape production', 'Estimated grape production (wine-equivalent)', 'Grape area', 'Wine consumption', 'Wine production']
unique world: ['New World', 'Old World']

top countries by map_value, per metri

In [52]:
master.to_csv(TABLEAU / "tableau_global_wine_map_master.csv", index=False)

saved one clean master file -> Tableau maps it all at the country level, with everything else as slicers.

#### geo-encoding for region 

In [61]:
from geopy.geocoders import ArcGIS

ROOT = Path.cwd()
TABLEAU = ROOT / "tableau"
region_path = ROOT / "data" / "processed" / "region_summary.csv"
cache_path = TABLEAU / "wine_region_geocoding_cache.csv"

regions = pd.read_csv(region_path)

# double checking formatting is all good
regions["country"] = regions["country"].astype(str).str.strip()
regions["region"] = regions["region"].astype(str).str.strip()
regions["total_area_ha"] = pd.to_numeric(regions["total_area_ha"], errors="coerce")

regions = regions[
    (regions["region"].str.lower() != "nan") &
    (regions["region"].str.lower() != "all") &
    (regions["region"].str.len() > 0)
].copy()

regions = (
    regions
    .sort_values("total_area_ha", ascending=False)
    [["country", "region", "total_area_ha"]]
    .drop_duplicates(["country", "region"])
    .head(TOP_N)
)

TOP_N = 1000  

# loading existing cache
if cache_path.exists():
    cache = pd.read_csv(cache_path)
else:
    cache = pd.DataFrame(columns=[
        "country", "region", "latitude", "longitude", "display_name", "geocode_status"
    ])

done = set(zip(cache["country"], cache["region"]))

# geocoder
geolocator = ArcGIS(timeout=10)

new_rows = []

for _, row in regions.iterrows():
    country = row["country"]
    region = row["region"]

    if (country, region) in done:
        continue

    query = f"{region}, {country}"
    print("Geocoding:", query)

    try:
        loc = geolocator.geocode(query)
    except Exception:
        loc = None

    new_rows.append({
        "country": country,
        "region": region,
        "latitude": loc.latitude if loc else None,
        "longitude": loc.longitude if loc else None,
        "display_name": loc.address if loc else None,
        "geocode_status": "matched" if loc else "not_found"
    })

# save cache
if new_rows:
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)

cache.to_csv(cache_path, index=False)

print("Saved:", cache_path)
print(cache["geocode_status"].value_counts(dropna=False))
cache.head(4)

Saved: C:\Users\jtrob\OneDrive\Desktop\wine-market-project\tableau\wine_region_geocoding_cache.csv
geocode_status
matched      1010
not_found     121
Name: count, dtype: int64


,country,region,latitude,longitude,display_name,geocode_status
0,Spain,Castilla La Mancha,39.417790,-2.623233,"Castilla-La Mancha, España",matched
1,Spain,Castilla-La Mancha,39.417790,-2.623233,"Castilla-La Mancha, España",matched
2,France,Languedoc Roussillon,43.654203,3.674670,"Languedoc-Roussillon, France métropolitaine, F...",matched
3,United States,California,36.701463,-118.755997,"California, United States",matched


In [60]:
master = pd.read_csv(TABLEAU / "tableau_global_wine_map_master.csv")
geo = pd.read_csv(TABLEAU / "wine_region_geocoding_cache.csv")

master_geo = master.merge(
    geo[["country", "region", "latitude", "longitude", "display_name", "geocode_status"]],
    on=["country", "region"],
    how="left"
)

master_geo.to_csv(TABLEAU / "tableau_global_wine_map_master_geocoded.csv", index=False)

print("Saved:", TABLEAU / "tableau_global_wine_map_master_geocoded.csv")
print("Rows with coordinates:", master_geo["latitude"].notna().sum())
print("Rows missing coordinates:", master_geo["latitude"].isna().sum())

Saved: C:\Users\jtrob\OneDrive\Desktop\wine-market-project\tableau\tableau_global_wine_map_master_geocoded.csv
Rows with coordinates: 196321
Rows missing coordinates: 34505


#### This is the end of this large notebook of cleaning, organizing, and validating the various data sources used in this project -- woohoo!